# Quantum Programming Lab3

In [1]:
from qiskit import *
from qiskit_aer import AerSimulator
import numpy as np
from qiskit.quantum_info import Operator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

## Part 1: Quantum Circuit for Shor's Algorithm

In this part, you will **construct a practical quantum circuit** for Shor’s algorithm.  

---

### 🔎 Background: How Shor’s Algorithm Works
Shor’s algorithm factors an integer $ N $ in three steps:

1. **Pick a co-prime $ a $:**  
   Choose a random integer $ a \in [2, N-1] $ such that $\gcd(a, N) = 1$.

2. **Find the order $ r $:**  
   Compute the smallest positive integer $ r $ such that  
   $$
   a^r \equiv 1 \pmod{N}.
   $$ 
   👉 This step is the **quantum part** (order finding).

3. **Compute a factor of $ N $:**  
   Use $ r $ to evaluate  
   $$
   \gcd(a^{r/2} + 1, N),
   $$  
   which gives a non-trivial factor of $ N $.

In this lab, we focus only on **Step 2: quantum order-finding**.

---

### 📌 Connection to Lecture
This lab builds on:  
- **Lecture X, Slides Y–Z** (Order finding and Shor’s Algorithm)  
- [Qiskit Textbook, Ch. 3.9](https://github.com/Qiskit/textbook/blob/main/notebooks/ch-algorithms/shor.ipynb)  

---

### 🧩 The Circuit Size
To factorize $ N $, the order-finding circuit requires:  
- $ m = \lceil \log_2(N) \rceil $ **computational qubits**  
- $ t = 2m $ **counting qubits**  
- Total = $ m + t = 3m $ qubits (minimum)

For our example $ N = 15 $:  
- $ m = 4 $  
- $ t = 8 $  
- Total = **12 qubits**

In the textbook implementation, 12 qubits and a deep cascade of controlled modular exponentiation gates are used:
$$
|x\rangle|0\rangle \;\mapsto\; |x\rangle|a^x \bmod N\rangle,
$$
creating the entangled state
$$
\sum_{x=0}^{2^m-1} |x\rangle |a^x \bmod N\rangle.
$$

⚠️ This design produces a **very deep circuit** that is not practical on today’s noisy quantum devices.

---

### ✂️ Reducing the Circuit
We can simplify the design for the special case $ a=7, N=15 $:  
- Only a few modular multiplications are needed.  
- These can be **hard-coded** instead of implementing a general-purpose modular exponentiation circuit.  
- This makes the circuit **shallower** while still capturing the essential behavior of order-finding.

---

### 📋 Your Task in Part 1
1. **Build the circuit:**  
   Construct a quantum order-finding circuit for $ a=7, N=15 $.  

2. **Verify the circuit**  
   Verify the correctness of your circuit.

3. **Output the circuit diagram:**  
   Print the circuit diagram and explain (2–3 sentences) how your design reduces depth.  

---

### 💡 Hints
- You don’t need a *general-purpose* modular exponentiation circuit — **hard-code the multiplications** for $ a=7, N=15 $.  
- Try computing the sequence $ 7^x \bmod 15 $ for small $ x $ first to see what gates you actually need.  
- Use `QuantumCircuit.draw("mpl")` to visualize your circuit and check its size.

---

### ⚠️ Common Mistakes
- **Overbuilding:** Don’t implement the full modular exponentiation for all cases — it’s too large and unnecessary. Try to use smarter method and build simpler circuit. 
- **Ignoring qubit roles:** Remember: counting register = phase estimation, computational register = modular arithmetic. Keep them separate.  

![title](../Figure/L7_Circ_gen.svg)

### 1. Removing Redundancy in Shor’s Circuit

The full Shor circuit uses a **general modular exponentiation operator** $ U $, applied in many controlled powers $ U^{2^k} $.  
For $ a=7, N=15 $, this is unnecessarily complex — we can simplify by focusing on the **specific modular multiplication**:

$$
U : |x\rangle \;\mapsto\; |7x \bmod 15\rangle.
$$

By hard-coding this operator, we avoid building a large, general-purpose unitary and **remove redundancy** in the circuit design.

---

### 🛠️ Step A: Construct the Gate $ U $

Run the following cell to define the gate $ U $ that implements  
$$
U|x\rangle = |(7x) \bmod 15\rangle.
$$

This gate will serve as the **building block** for the order-finding part of Shor’s algorithm.

In [11]:
## Create 7mod15 gate
N = 15
m = int(np.ceil(np.log2(N)))

U_qc = QuantumCircuit(m)
U_qc.x(range(m))
U_qc.swap(1, 2)
U_qc.swap(2, 3)
U_qc.swap(0, 3)

U = U_qc.to_gate()
U.name ='{}Mod{}'.format(7, N)

### 2. Verify the Unitary Operator $ U $

Now that we have defined the unitary $ U $ for the mapping

$$
U|x\rangle = |(7x) \bmod 15\rangle,
$$

let’s **test it on sample inputs** to confirm it works correctly.

---

### 🛠️ Step B: Build a Test Circuit (10 points)
1. Use $ m = 4 $ qubits to represent numbers from 0 to 15.  
2. Prepare an input state $|x\rangle$ for some chosen integer $ x $.  
   - Example inputs:  
     - $|1\rangle = |0001\rangle$  
     - $|5\rangle = |0101\rangle$  
     - $|13\rangle = |1101\rangle$  
3. Apply the $ U $ gate.  
4. Measure the output and confirm that it matches the expected mapping.

---

### ✅ Expected Outcomes
- Input $|1\rangle \;\mapsto\; |7\rangle = |0111\rangle$  
- Input $|13\rangle \;\mapsto\; |1\rangle = |0001\rangle$  
- (Try a few other inputs of your choice!)

---

In [3]:
## Your code here

### 🛠️ Step B: Verify $ U^{2^2} = I $ (10 points)

1. Build a 4-qubit circuit that applies the $ U $ gate **4 times in sequence** (since $ 2^2 = 4 $).  
2. Use the `Operator` class (imported earlier) to obtain the **matrix representation** of this circuit.  
3. Compare the result with the identity matrix.  
   - If $ U^{4} = I $, then higher powers of $ U $ will also cycle back to the identity.  
   - This reveals that applying $ U^{2^n} $ for $ n > 1 $ is **redundant** in this special case.

---

In [4]:
## Your code here

### 4. Run the Reduced Circuit: Quantum Phase Estimation (QPE)

Now that we know many powers of $ U $ are redundant, we can build a **reduced version** of Shor’s order-finding circuit.  
This circuit is called **`shor_QPE`** because it uses **Quantum Phase Estimation (QPE)** to extract the order.

---

### 🛠️ Step C: Execute the Reduced Circuit (10 points)
1. Run the cell below to construct the reduced circuit `shor_QPE`.  
   - It uses only the necessary powers of $ U $ (up to $ U^2 $), making it shallower than the full textbook version.  
2. Execute the circuit on the **Aersimulator**.  
3. Compare the results against the textbook phase estimation results from the [Qiskit Textbook, Ch. 3.9](https://github.com/Qiskit/textbook/blob/main/notebooks/ch-algorithms/quantum-phase-estimation.ipynb).

---

In [10]:
def cU_multi(k):
    circ = QuantumCircuit(m)
    for _ in range(2**k):
        circ.append(U, range(m))
    
    U_multi = circ.to_gate()
    U_multi.name = '7Mod15_[2^{}]'.format(k)
    
    cU_multi = U_multi.control()
    return cU_multi

In [4]:
def qft(n):
    """Creates an n-qubit QFT circuit"""
    circuit = QuantumCircuit(n)
    def swap_registers(circuit, n):
        for qubit in range(n//2):
            circuit.swap(qubit, n-qubit-1)
        return circuit
    def qft_rotations(circuit, n):
        """Performs qft on the first n qubits in circuit (without swaps)"""
        if n == 0:
            return circuit
        n -= 1
        circuit.h(n)
        for qubit in range(n):
            circuit.cp(np.pi/2**(n-qubit), qubit, n)
        qft_rotations(circuit, n)
    
    qft_rotations(circuit, n)
    swap_registers(circuit, n)
    return circuit

In [ ]:
# QPE circuit for Shor
t = 3 
shor_QPE = QuantumCircuit(t+m, t)
shor_QPE.h(range(t))

shor_QPE.x(t)
for idx in range(t-1):
    shor_QPE.append(cU_multi(idx), [idx]+ list(range(t,t+m)))

qft_dag = qft(t).inverse()
qft_dag.name = 'QFT+'

shor_QPE.append(qft_dag, range(t))
shor_QPE.measure(range(t), range(t))

shor_QPE.draw()

In [ ]:
backend = AerSimulator()

transpiled_circuit = transpile(shor_QPE, backend)
job = backend.run(transpiled_circuit, shots=1000, memory=True)
count_QPE= job.result().get_counts()

key_new = [str(int(key,2)/2**3) for key in count_QPE.keys()]
count_new_QPE = dict(zip(key_new, count_QPE.values()))
plot_histogram(count_new_QPE)

## Part 2: Noisy simulation of the quantum order-finding circuits.(10 points)

### 🎯 Goal
Compare the **noise robustness** of two order-finding circuits:
- `shor_Orig`  — textbook-style (deeper)
- `shor_QPE`   — reduced / redundancy-removed (shallower)

You will:
1. Build (or import) realistic **noise models** / fake backends.
2. Run both circuits under **noiseless** and **noisy** simulation.
3. Quantify and compare outcome **quality** with simple metrics.
4. Briefly explain why this motivates **quantum error correction (QEC)** for scaling.

> 📖 Optional reading: *<a href="https://quantum-journal.org/papers/q-2021-04-15-433/">How to factor 2048 bit RSA integers in 8 hours using 20 million noisy qubits</a>* by Craig Gidney. You’ll see how noise and depth limit practical algorithms and why QEC is essential.

---

### 🧱 Setup Checklist
- You should already have constructed both circuits in Part 1:
  - `shor_Orig` (12 qubits, deep)
  - `shor_QPE`  (12 qubits, reduced powers of $U$)
- Counting register width $t = 2m$; for $N=15$, $m=4, t=8$.

---

### 🔎 What counts as “success” here?
For $a=7, N=15$, the **order is $r=4$**. QPE ideally yields phases $k/r \in \{0,\tfrac14,\tfrac12,\tfrac34\}$.
With an $t$-qubit counting register, measurement bitstrings correspond to integers $y \in \{0,\dots,2^t\!-\!1\}$ and phases $y/2^t$.
So, a shot is a **hit** if $y/2^t$ is closest to one of $\{0,\tfrac14,\tfrac12,\tfrac34\}$ (ties allowed).

We’ll report:
- **Hit-rate** (fraction of shots landing closest to $\{0,\tfrac14,\tfrac12,\tfrac34\}$)
- (Optional) **KL divergence** or **total-variation distance** vs the ideal distribution

> ⚠️ Bit-endianness note: if your counts look “shifted,” check whether the most-significant bit is left or right in your measurement map.

---

### 🛠️ Step D — Baseline (Noiseless) Reference
Run both circuits on an ideal simulator to establish a clean reference.

In [ ]:
t = 2*m

shor_Orig = QuantumCircuit(t+m, t)
shor_Orig.h(range(t))

shor_Orig.x(t)
for idx in range(t):
    shor_Orig.append(cU_multi(idx), [idx]+ list(range(t,t+m)))

qft_dag = qft(t).inverse()
qft_dag.name = 'QFT+'

shor_Orig.append(qft_dag, range(t))
shor_Orig.measure(range(t), range(t))
    
shor_Orig.draw()

In [ ]:
backend = AerSimulator()

transpiled_circuit = transpile(shor_Orig, backend)
job = backend.run(transpiled_circuit, shots=1000, memory=True)
count_Orig= job.result().get_counts()

key_new = [str(int(key,2)/2**t) for key in count_Orig.keys()]
count_Orig = dict(zip(key_new, count_Orig.values()))
plot_histogram(count_Orig, title='textbook circuit simulation result No noise')

### 2. Noise Sweep: Compare `shor_Orig` vs `shor_QPE` (10 points)

#### 🎯 Task
Using the **provided noise model** (parameterized by error strength $p$), run both circuits over  
$$
p \in \{0.1, 0.2, 0.3, 0.4, 0.5\}
$$
and **compare their results to the noiseless case** ($p=0$).

You will:
- Execute `shor_Orig` and `shor_QPE` with each $p$.
- Compute a **hit-rate** metric (fraction of shots closest to phases $\{0,\tfrac14,\tfrac12,\tfrac34\}$).
- Plot **hit-rate vs $p$** for both circuits with the $p=0$ baseline.

> ℹ️ The notebook earlier should define or import a helper to build the noise model. If not, a minimal fallback is included below.

In [38]:
# Import from Qiskit Aer noise module
from qiskit_aer.noise import (NoiseModel, QuantumError, ReadoutError,
    pauli_error, depolarizing_error, thermal_relaxation_error)


def construct_bitphaseflip_noise_model(p):
    # We set three different noise parameter the same as p
    p_reset = p
    p_meas = p
    p_gate1 = p

    #Phase flip noise when there is gate of measurement
    error_reset = pauli_error([('Z', p_reset), ('I', 1 - p_reset)])
    error_meas = pauli_error([('Z',p_meas), ('I', 1 - p_meas)])
    error_gate1 = pauli_error([('Z',p_gate1), ('I', 1 - p_gate1)])
    error_gate2 = error_gate1.tensor(error_gate1)

    #Bitflip noise when there is gate of measurement
    error_reset_bit = pauli_error([('X', p_reset), ('I', 1 - p_reset)])
    error_meas_bit = pauli_error([('X',p_meas), ('I', 1 - p_meas)])
    error_gate1_bit = pauli_error([('X',p_gate1), ('I', 1 - p_gate1)])
    error_gate2_bit = error_gate1_bit.tensor(error_gate1)

    # Add above errors to the same noise model object
    noise_bitphase_flip = NoiseModel()
    noise_bitphase_flip.add_all_qubit_quantum_error(error_reset_bit, "reset")
    noise_bitphase_flip.add_all_qubit_quantum_error(error_meas_bit, "measure")
    noise_bitphase_flip.add_all_qubit_quantum_error(error_gate1_bit, ["x","h","t","tdg"])
    noise_bitphase_flip.add_all_qubit_quantum_error(error_gate2_bit, ["cx,ccx"])
    

    return noise_bitphase_flip

Setup your noisemodel with noise parameter, and run ths noisy simulation

In [43]:
p=0
noisemodel=construct_bitphaseflip_noise_model(p)

In [ ]:
backend = AerSimulator()

transpiled_circuit = transpile(shor_Orig, backend)
job = backend.run(transpiled_circuit,noise_model=noisemodel, shots=1000, memory=True)
count_Orig= job.result().get_counts()

key_new = [str(int(key,2)/2**t) for key in count_Orig.keys()]
count_Orig = dict(zip(key_new, count_Orig.values()))
plot_histogram(count_Orig, title='textbook circuit simulation with noise')

## Additional Task: Repeat the Lab for `N = 21` and `N = 63` (20 points)

### 🎯 Goal
Replicate the **Part 1 + Part 2** workflow for two new composites and compare their behavior to `N = 15`:

- Case A: $N = 21$  ($\Rightarrow m = \lceil \log2 21 \rceil = 5$, counting $t = 2m = 10$, total qubits ≈ 15)
- Case B: $N = 63$  ($\Rightarrow m = \lceil log2 63\rceil = 6 $, counting `t = 2m = 12`, total qubits ≈ 18)

You’ll:
1. Choose a **co-prime** base `a` (e.g., `a = 2` works for both 21 and 63).
2. Build the **modular-multiplication** unitary $U : \ket{x} \rightarrow \ket{a \times x \mod N}$ on `m` qubits.
3. **Verify powers of U** (find redundancy pattern from the order $r = ord\_N(a)$).
4. Construct a reduced **QPE-style** order-finding circuit (`shor_QPE`) using only the **needed** controlled powers.
5. Run **ideal** and **noisy** simulations; compare to the $N = 15$ case.
6. Describe **major differences** you observe.

> ℹ️ Tip: Keep `a` small and co-prime to `N`. For both `N=21` and `N=63`, popular safe picks are $a \in \{2,5,10,11,17,19,...\}$ with $gcd(a,N)=1$.

---

### ⚠️ You are not allowed to build the modular multiplication $U$ directly by matrix
In this homework, you are asked to compile modular multiplication into single qubit gates(e.g. X,Y,Z,RZ,etc.) and two qubit gates(e.g. swap, cnot, etc.). Any method that directly construct a virtual unitary from matrix is not allowed. 


In Shor’s order-finding circuit we need controlled applications of powers of the modular multiplication operator $U$.  
Specifically, the phase estimation routine requires controlled-$U^{2^k}$ for $k = 0,1,\dots,t-1$.

To make this reusable, we define a helper function **`_cU_multi(U, m, k)`** that:

1. Takes as input:
   - `U`: the base modular multiplication gate (on `m` qubits),
   - `m`: number of qubits in the target register,
   - `k`: the exponent index.
2. Builds the gate $U^{2^k}$ by repeating $U$ exactly $2^k$ times.
3. Converts this into a controlled gate so it can be applied with one counting-register qubit as the control.

This modular design makes it easy to plug different $U$ gates into the phase estimation circuit, whether for $N=15$, $21$, or $63$.

In [53]:
def _cU_multi(U,m,k):
    circ = QuantumCircuit(m)
    for _ in range(2**k):
        circ.append(U, range(m))
    
    U_multi = circ.to_gate()
    U_multi.name = '7Mod15_[2^{}]'.format(k)
    
    cU_multi = U_multi.control()
    return cU_multi

### Example: Shor15

Below is an example implementation of the **Shor15** function.  
This shows the **exact format we expect you to follow** when building your own order-finding circuits for $N=21$ and $N=63$.  

Notes:
- You may choose any base $a$ that is coprime to $N$.
- The function should:
  1. Define the modular multiplication gate $U$,
  2. Use `_cU_multi` to add the controlled powers of $U$,
  3. Apply the inverse QFT on the counting register,
  4. Measure and return `(a, shor_QPE, count_QPE)`.

In [54]:
def Shor15():
    ## Code for constructing shor_QPE for factoring 15 starts here
    a = 7
    N = 15
    m = int(np.ceil(np.log2(N)))
    
    U_qc = QuantumCircuit(m)
    U_qc.x(range(m))
    U_qc.swap(1, 2)
    U_qc.swap(2, 3)
    U_qc.swap(0, 3)
    
    U = U_qc.to_gate()
    U.name ='{}Mod{}'.format(7, N)

    # QPE circuit for Shor15
    t = 3 
    shor_QPE = QuantumCircuit(t+m, t)
    shor_QPE.h(range(t))
    
    shor_QPE.x(t)
    for idx in range(t-1):
        shor_QPE.append(_cU_multi(U,m,idx), [idx]+ list(range(t,t+m)))
    
    qft_dag = qft(t).inverse()
    qft_dag.name = 'QFT+'
    
    shor_QPE.append(qft_dag, range(t))
    shor_QPE.measure(range(t), range(t))
    ## Code for constructing shor_QPE for factoring 15 ends here    

    
    backend = AerSimulator()
    transpiled_circuit = transpile(shor_QPE, backend)
    job = backend.run(transpiled_circuit, shots=1000, memory=True)
    count_QPE= job.result().get_counts()
    return a, shor_QPE, count_QPE

Please implement the following function of factoring 21. 
You should submit this function in .py file.

In [55]:
def Shor21():
    ##Your circuit for creating quantum circuit shor_QPT of factoring 21 starts here
    a = 2 # By default, you can use a=2
    shor_QPE=None




    
    ##Your circuit for creating quantum circuit shor_QPT of factoring 21 ends here    
    backend = AerSimulator()
    transpiled_circuit = transpile(shor_QPE, backend)
    job = backend.run(transpiled_circuit, shots=1000, memory=True)
    count_QPE= job.result().get_counts()
    return a, shor_QPE, count_QPE

Please implement the following function of factoring 21. 
You should submit this function in .py file.

In [56]:
def Shor63():
    ##Your circuit for creating quantum circuit shor_QPT of factoring 63 starts here
    shor_QPE=None
    a = 2 # By default, you can use a=2    




    
    ##Your circuit for creating quantum circuit shor_QPT of factoring 63 ends here    
    backend = AerSimulator()
    transpiled_circuit = transpile(shor_QPE, backend)
    job = backend.run(transpiled_circuit, shots=1000, memory=True)
    count_QPE= job.result().get_counts()
    return a, shor_QPE, count_QPE

---
### ✅ Testing your solution

You can validate your implementation with the provided test script:

👉 **[testShor.py](https://github.com/yezhuoyang/QuantumLab/blob/main/test/testShor.py)**

**How to run**

First, copy your implementation of Shor21, Shor63 function to testShor.py, then execute:

```bash
# from the repository root folder
python test/testShor.py
```